In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# groups
dynamic_names = ["ca", "ae", "al", "am", "ce", "el", "cl", "cm", "cp"]

final_fusion_results = []

name_mapping = {
    "ca": "CN vs. AD",
    "ae": "AD vs. EMCI",
    "al": "AD vs. LMCI",
    "am": "AD vs. MCI",
    "ce": "CN vs. EMCI",
    "el": "EMCI vs. LMCI",
    "cl": "CN vs. LMCI",
    "cm": "CN vs. MCI",
    "cp": "CN vs. PT"
}

for name in dynamic_names:
    try:
        print(f"🔄 Processing Group: {name}")

        exp_samples_path = f"all_matrics/{name}/samples_{name}.csv"
        exp_voavg_path = f"all_matrics/{name}/voavg_matrix_{name}.csv"
        met_samples_path = f"all_matrics/{name}/m/samples_m_{name}.csv"
        met_voavg_path = f"all_matrics/{name}/m/voavg_matrix_m_{name}.csv"

        voavg_exp = pd.read_csv(exp_voavg_path)
        samples_exp = pd.read_csv(exp_samples_path)
        voavg_met = pd.read_csv(met_voavg_path)
        samples_met = pd.read_csv(met_samples_path)

        voavg_exp['Classifier_Count'] = voavg_exp['Combination'].apply(lambda x: len(x.split(',')))
        voavg_met['Classifier_Count'] = voavg_met['Combination'].apply(lambda x: len(x.split(',')))

        filtered_exp = voavg_exp[voavg_exp["Classifier_Count"] > 2]
        filtered_met = voavg_met[voavg_met["Classifier_Count"] > 2]

        best_exp_auc = filtered_exp.loc[filtered_exp["AUC"].idxmax()]
        best_exp_auc_f = filtered_exp.loc[filtered_exp["AUC_f"].idxmax()]
        best_met_auc = filtered_met.loc[filtered_met["AUC"].idxmax()]
        best_met_auc_f = filtered_met.loc[filtered_met["AUC_f"].idxmax()]

        # select the best auc for each omics
        if best_exp_auc["AUC"] >= best_exp_auc_f["AUC_f"]:
            best_exp = best_exp_auc
            exp_pred_col = "Prediction"
        else:
            best_exp = best_exp_auc_f
            exp_pred_col = "Prediction_f"

        if best_met_auc["AUC"] >= best_met_auc_f["AUC_f"]:
            best_met = best_met_auc
            met_pred_col = "Prediction"
        else:
            best_met = best_met_auc_f
            met_pred_col = "Prediction_f"

        best_exp["Best_AUC"] = max(best_exp_auc["AUC"], best_exp_auc_f["AUC_f"])
        best_met["Best_AUC"] = max(best_met_auc["AUC"], best_met_auc_f["AUC_f"])

        exp_predictions = samples_exp[
            (samples_exp["Combination"] == best_exp["Combination"]) &
            (samples_exp["MetaModel"] == best_exp["MetaModel"])
        ][["SampleRID", exp_pred_col]].rename(columns={exp_pred_col: "exp_prediction"})

        met_predictions = samples_met[
            (samples_met["Combination"] == best_met["Combination"]) &
            (samples_met["MetaModel"] == best_met["MetaModel"])
        ][["SampleRID", met_pred_col]].rename(columns={met_pred_col: "met_prediction"})

        final_predictions = pd.merge(exp_predictions, met_predictions, on="SampleRID", how="inner")

        y_true = samples_exp[["SampleRID", "DX_bl"]].drop_duplicates(subset="SampleRID")
        y_true = y_true[y_true["SampleRID"].isin(final_predictions["SampleRID"])]
        y_true = final_predictions[["SampleRID"]].merge(y_true, on="SampleRID", how="left")["DX_bl"]

        # optimizing the best weight for integration
        alphas = np.round(np.arange(0, 1.01, 0.01), 2)
        fusion_results = []

        for alpha in alphas:
            final_predictions["fusion_score"] = alpha * final_predictions["exp_prediction"] + (1 - alpha) * final_predictions["met_prediction"]
            auc = roc_auc_score(y_true, final_predictions["fusion_score"])
            fusion_results.append([alpha, auc])

        fusion_df = pd.DataFrame(fusion_results, columns=["Alpha", "AUC"])

        best_row = fusion_df.loc[fusion_df["AUC"].idxmax()]
        best_alpha = best_row["Alpha"]
        best_auc = best_row["AUC"]

        # plt the best AUC
        os.makedirs(f"all_matrics/{name}", exist_ok=True)

        min_auc = fusion_df["AUC"].min() - 0.01
        max_auc = fusion_df["AUC"].max() + 0.015

        plt.figure(figsize=(6, 6))
        plt.axhline(y=best_exp["Best_AUC"], color="#4682B4", linestyle="--", linewidth=3, label=f"AUC (Exp) - {best_exp['Best_AUC']:.2f}")
        plt.axhline(y=best_met["Best_AUC"], color="#2E8B57", linestyle="dotted", linewidth=3, label=f"AUC (Met) - {best_met['Best_AUC']:.2f}")
        plt.plot(fusion_df["Alpha"], fusion_df["AUC"], 'o-', color="#EE82EE", markersize=1, linewidth=4, label="AUC (Int)")

        plt.axvline(x=best_alpha, color="orange", linestyle="--", linewidth=2)
        plt.annotate(f"Max AUC: {best_auc:.2f}\nα: {best_alpha}",
                    xy=(best_alpha, best_auc), 
                    xytext=(best_alpha + 0.1, best_auc - 0.03),
                    fontsize=20, fontweight='bold')

        plt.ylim(min_auc, max_auc)
        plt.xticks(fontsize=18, fontweight='bold')
        plt.yticks(fontsize=18, fontweight='bold')

        custom_title = name_mapping.get(name, name)
        plt.title(f"{custom_title}", fontsize=20, fontweight='bold')
        plt.legend(loc='upper left', fontsize=16, prop={'weight': 'bold'})
        
        plt.grid(False)
        plt.show()

        print(f"Done Processing Group: {name}")

    except Exception as e:
        print(f"Processing {name} with Error: {e}")

print("All Groups Done！")
